In [ ]:
# === Cell 0: Env sanity ===
import torch, transformers, datasets, evaluate, platform, sys
print("Python:", sys.version)
print("OS:", platform.platform())
print("PyTorch:", torch.__version__, "| CUDA avail:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.memory_summary(abbreviated=True))

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)


In [10]:
# === Cell 1: Constants & paths ===
from pathlib import Path
import os, random, numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
import torch
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    try:
        torch.set_float32_matmul_precision("high")
        torch.backends.cuda.matmul.allow_tf32 = True
    except Exception:
        pass

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Device={device} | bf16={use_bf16} | fp16={use_fp16}")

PROJECT_DIR = "./nllb_bilingual_en_es_pt"
os.makedirs(PROJECT_DIR, exist_ok=True)

ROOT = Path(r"J:\FINAL PROJECT")   # <- your root
TMX_ES = ROOT / "data" / "en-es.tmx"
TMX_PT = ROOT / "data" / "en-pt.tmx"
print("exists:", TMX_ES.exists(), TMX_PT.exists())

MAX_SAMPLES = 1200
EVAL_SAMPLES = 300
MAX_SRC_LEN = 64      # was 64
MAX_TGT_LEN = 64   # was 64
BATCH = 8
GRAD_ACCUM = 1
MAX_STEPS = 500


Device=cuda | bf16=True | fp16=False
exists: True True


In [11]:
# === Cell 2: TMX loader + sample peek ===
import io, gzip, unicodedata, regex as re
from xml.etree import ElementTree as ET
from datasets import Dataset

def _nfc_ws(s: str) -> str:
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("\u00A0"," ").replace("\u202F"," ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _norm_lang(tag: str) -> str:
    if not tag: return ""
    t = tag.replace("_","-").lower()
    if t.startswith("eng"): return "en"
    if t.startswith("spa"): return "es"
    if t.startswith("por"): return "pt"
    return t.split("-")[0]

def _open_maybe_gz(path: str):
    return io.TextIOWrapper(gzip.open(path, "rb"), encoding="utf-8", errors="ignore") if path.endswith(".gz") \
           else io.open(path, "r", encoding="utf-8", errors="ignore")

def load_tmx_as_dataset(tmx_path: str, src="en", tgt="es", max_samples: int=None, seed: int=42) -> Dataset:
    p = Path(tmx_path); assert p.exists(), f"Missing TMX: {tmx_path}"
    src = _norm_lang(src); tgt = _norm_lang(tgt)
    pairs = []
    with _open_maybe_gz(str(p)) as fh:
        context = ET.iterparse(fh, events=("start","end"))
        _, root = next(context)
        cur_src, cur_tgt = None, None
        for event, elem in context:
            tag = elem.tag.lower()
            if event == "start" and tag.endswith("tu"):
                cur_src, cur_tgt = None, None
            if event == "end" and tag.endswith("tuv"):
                lang = _norm_lang(elem.attrib.get("{http://www.w3.org/XML/1998/namespace}lang", elem.attrib.get("lang","")))
                seg = next((c for c in elem if c.tag.lower().endswith("seg")), None)
                if seg is not None:
                    text = _nfc_ws("".join(seg.itertext()))
                    if lang == src and text: cur_src = text
                    elif lang == tgt and text: cur_tgt = text
            if event == "end" and tag.endswith("tu"):
                if cur_src and cur_tgt:
                    pairs.append((cur_src, cur_tgt))
                root.clear()
                if max_samples and len(pairs) >= max_samples:
                    break
    if max_samples and len(pairs) > max_samples:
        import random
        rnd = random.Random(seed)
        idx = list(range(len(pairs))); rnd.shuffle(idx); idx = idx[:max_samples]
        pairs = [pairs[i] for i in idx]
    return Dataset.from_dict({"src_text":[s for s,_ in pairs], "tgt_text":[t for _,t in pairs]})

print("⏳ Loading TMX…")
ds_es_all = load_tmx_as_dataset(str(TMX_ES), src="en", tgt="es",
                                max_samples=MAX_SAMPLES+EVAL_SAMPLES, seed=SEED).shuffle(seed=SEED)
ds_pt_all = load_tmx_as_dataset(str(TMX_PT), src="en", tgt="pt",
                                max_samples=MAX_SAMPLES+EVAL_SAMPLES, seed=SEED).shuffle(seed=SEED)

print("Samples EN→ES:", [tuple(ds_es_all[i].values()) for i in range(3)])
print("Samples EN→PT:", [tuple(ds_pt_all[i].values()) for i in range(3)])


⏳ Loading TMX…
Samples EN→ES: [('The disaccharidases, peptidases, alkaline phosphatases, adenosine triphosphatases and esterases are reduced.', 'Las disacaridasas, peptidasas, la fosfatasa alcalina, la adenosina trifosfatasa y las esterasas están disminuidas.'), ('It was also found hepatosplenomegaly and infection by Epstein Barr virus.', 'Se encontró además hepatoesplenomegalia e infección por virus del Epstein Barr.'), ('These patients show significant increases in Lactobacillus and Veillonella. It has also been shown that the production of acetic and propionic acid is related to presence of greater numbers and intensity of symptoms such as abdominal pain, bloating, and changes in bowel habits resulting in decreased quality of life 34, 35.', 'Estos pacientes mostraron un significativo aumentos de Lactobacillus y Veillonella y además que la producción de ácido acético y propiónico estuvo relacionada con aumento de los síntomas dolor abdominal, "Bloating" y variación del hábito intesti

In [12]:

# ------------------------
# LIGHT/SAFE CLEANING (EN→ES)
# -------------------------
CLEAN_ES = True
if CLEAN_ES:
    import regex as re, unicodedata
    import langid
    langid.set_languages(['en','es'])  # we only check source 'en' here

    URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
    PUNCT_ONLY_RE = re.compile(r"^\p{P}+$", re.UNICODE)

    def nfc(s):
        s = unicodedata.normalize("NFC", s or "")
        s = s.replace("\u00A0"," ").replace("\u202F"," ")
        return re.sub(r"\s+", " ", s).strip()

    def clean_example(ex):
        s = URL_RE.sub("<URL>", nfc(ex["src_text"]))
        t = URL_RE.sub("<URL>", nfc(ex["tgt_text"]))
        return {"src_text": s, "tgt_text": t}

    def basic_quality_ok(ex):
        s, t = ex["src_text"], ex["tgt_text"]
        if len(s.split()) < 2 or len(t.split()) < 2: return False
        if PUNCT_ONLY_RE.match(s) or PUNCT_ONLY_RE.match(t): return False
        if len(s.split()) > 200 or len(t.split()) > 200: return False
        ls, cs = langid.classify(s)
        if ls != "en" and cs > 0.7: return False
        return True

    def dedup_pairs(ds):
        seen, keep = set(), []
        for i, ex in enumerate(ds):
            key = (ex["src_text"].lower(), ex["tgt_text"].lower())
            if key not in seen:
                seen.add(key); keep.append(i)
        return ds.select(keep)

    print("[ES] pre-clean size:", len(ds_es_all))
    ds_es_all = ds_es_all.map(clean_example)
    ds_es_all = dedup_pairs(ds_es_all)
    ds_es_all = ds_es_all.filter(basic_quality_ok)
    print("[ES] post-clean size:", len(ds_es_all))

# -------------------------
# LIGHT/SAFE CLEANING (EN→PT)
# -------------------------
CLEAN_PT = True
if CLEAN_PT:
    import regex as re, unicodedata
    import langid
    langid.set_languages(['en','pt'])  # we only check source 'en' here

    URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
    PUNCT_ONLY_RE = re.compile(r"^\p{P}+$", re.UNICODE)

    def nfc_ws(s: str) -> str:
        s = unicodedata.normalize("NFC", s or "")
        s = s.replace("\u00A0"," ").replace("\u202F"," ")
        return re.sub(r"\s+", " ", s).strip()

    def clean_example_pt(ex):
        s = URL_RE.sub("<URL>", nfc_ws(ex["src_text"]))
        t = URL_RE.sub("<URL>", nfc_ws(ex["tgt_text"]))
        return {"src_text": s, "tgt_text": t}

    def basic_quality_ok_pt(ex):
        s, t = ex["src_text"], ex["tgt_text"]
        if len(s.split()) < 2 or len(t.split()) < 2: return False
        if PUNCT_ONLY_RE.match(s) or PUNCT_ONLY_RE.match(t): return False
        if len(s.split()) > 200 or len(t.split()) > 200: return False
        ls, cs = langid.classify(s)
        if ls != "en" and cs > 0.7: return False
        return True

    def dedup_pairs_pt(ds):
        seen, keep = set(), []
        for i, ex in enumerate(ds):
            key = (ex["src_text"].lower(), ex["tgt_text"].lower())
            if key not in seen:
                seen.add(key); keep.append(i)
        return ds.select(keep)

    print("[PT] pre-clean size:", len(ds_pt_all))
    ds_pt_all = ds_pt_all.map(clean_example_pt)
    ds_pt_all = dedup_pairs_pt(ds_pt_all)
    ds_pt_all = ds_pt_all.filter(basic_quality_ok_pt)
    print("[PT] post-clean size:", len(ds_pt_all))

# -------- Split AFTER cleaning --------
es_eval = min(EVAL_SAMPLES, max(50, int(0.10 * len(ds_es_all))))
pt_eval = min(EVAL_SAMPLES, max(50, int(0.10 * len(ds_pt_all))))

split_es = ds_es_all.train_test_split(test_size=es_eval, seed=SEED, shuffle=True)
split_pt = ds_pt_all.train_test_split(test_size=pt_eval, seed=SEED, shuffle=True)

train_es, valid_es = split_es["train"], split_es["test"]
train_pt, valid_pt = split_pt["train"], split_pt["test"]

print("Final split sizes:",
      f"EN→ES train {len(train_es)} / valid {len(valid_es)}  |  EN→PT train {len(train_pt)} / valid {len(valid_pt)}")


[ES] pre-clean size: 1500


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

[ES] post-clean size: 1498
[PT] pre-clean size: 1500


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1499 [00:00<?, ? examples/s]

[PT] post-clean size: 1492
Final split sizes: EN→ES train 1349 / valid 149  |  EN→PT train 1343 / valid 149


In [13]:
# === Cell 4: Tokenizers & models with robust language forcing ===
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nllb_ckpt = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
ES_LANG  = "spa_Latn"
PT_LANG  = "por_Latn"

def forced_bos_id(tok, lang_code: str):
    m = getattr(tok, "lang_code_to_id", None)
    if isinstance(m, dict) and lang_code in m:
        return m[lang_code]
    if hasattr(tok, "get_lang_id"):
        return tok.get_lang_id(lang_code)
    i = tok.convert_tokens_to_ids(lang_code)
    if i is None or i == getattr(tok, "unk_token_id", None):
        raise RuntimeError(f"Could not resolve id for {lang_code}. Try upgrading `transformers`.")
    return i

# ES
tok_es = AutoTokenizer.from_pretrained(nllb_ckpt, use_fast=True)
tok_es.src_lang = SRC_LANG
tok_es.tgt_lang = ES_LANG
mod_es = AutoModelForSeq2SeqLM.from_pretrained(nllb_ckpt)
_es_id = forced_bos_id(tok_es, ES_LANG)
for m in (mod_es.config, mod_es.generation_config):
    m.forced_bos_token_id = _es_id
    m.decoder_start_token_id = _es_id

# PT
tok_pt = AutoTokenizer.from_pretrained(nllb_ckpt, use_fast=True)
tok_pt.src_lang = SRC_LANG
tok_pt.tgt_lang = PT_LANG
mod_pt = AutoModelForSeq2SeqLM.from_pretrained(nllb_ckpt)
_pt_id = forced_bos_id(tok_pt, PT_LANG)
for m in (mod_pt.config, mod_pt.generation_config):
    m.forced_bos_token_id = _pt_id
    m.decoder_start_token_id = _pt_id

# (Optional) move to GPU
if device == "cuda":
    mod_es.to(device)
    mod_pt.to(device)

# try: mod_es.gradient_checkpointing_enable()
# except Exception: pass
# try: mod_pt.gradient_checkpointing_enable()
# except Exception: pass


In [14]:
# === Cell 5: Preprocess using the same tokenizers ===
from datasets import DatasetDict

def preprocess_any(batch, tok):
    model_inputs = tok(batch["src_text"], truncation=True, max_length=MAX_SRC_LEN)
    try:
        labels = tok(text_target=batch["tgt_text"], truncation=True, max_length=MAX_TGT_LEN)
    except TypeError:
        from contextlib import contextmanager
        cm = tok.as_target_tokenizer() if hasattr(tok, "as_target_tokenizer") else contextmanager(lambda: (yield))()
        with cm:
            labels = tok(batch["tgt_text"], truncation=True, max_length=MAX_TGT_LEN)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_es_tok = train_es.map(lambda b: preprocess_any(b, tok_es), batched=True, remove_columns=train_es.column_names)
valid_es_tok = valid_es.map(lambda b: preprocess_any(b, tok_es), batched=True, remove_columns=valid_es.column_names)
train_pt_tok = train_pt.map(lambda b: preprocess_any(b, tok_pt), batched=True, remove_columns=train_pt.column_names)
valid_pt_tok = valid_pt.map(lambda b: preprocess_any(b, tok_pt), batched=True, remove_columns=valid_pt.column_names)

print("Tokenized shapes ES:", train_es_tok[0].keys(), " | PT:", train_pt_tok[0].keys())


Map:   0%|          | 0/1349 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Map:   0%|          | 0/1343 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Tokenized shapes ES: dict_keys(['input_ids', 'attention_mask', 'labels'])  | PT: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [15]:
# === Turbo Cell 6: Fast generation + zero-shot sanity ===
import numpy as np
import evaluate as _evaluate
import torch

bleu_metric = _evaluate.load("sacrebleu")
chrf_metric = _evaluate.load("chrf")

# Fast presets
MAX_NEW_TOK  = 96     # was 128; increase later if needed
MIN_NEW_TOK  = 2
NUM_BEAMS    = 1      # <-- big speedup; set to 4 later for “final” metrics
BATCH_SIZE   = 128    # bump as high as VRAM allows (64/128/192)

# Make sure models are in eval mode and use_cache is on (Trainer may disable)
for _m in (mod_es, mod_pt):
    _m.eval()
    if getattr(_m.config, "use_cache", True) is False:
        _m.config.use_cache = True

def _autocast_if_cuda():
    if torch.cuda.is_available():
        # prefer bf16 if supported, otherwise fp16
        use_bf16 = getattr(torch.cuda, "is_bf16_supported", lambda: False)()
        return torch.autocast("cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16)
    # no-op context on CPU
    return torch.cuda.amp.autocast(enabled=False)

@torch.inference_mode()
def batch_generate_fast(model, tok, texts,
                        max_len_src=128, max_new=MAX_NEW_TOK, beams=NUM_BEAMS,
                        batch_size=BATCH_SIZE, pad_to_multiple_of=8):
    # Tokenize on CPU, pad to 8 for tensor cores, then move to device
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
              max_length=max_len_src, pad_to_multiple_of=pad_to_multiple_of)
    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}
    lang_id = forced_bos_id(tok, tok.tgt_lang)

    preds = []
    with _autocast_if_cuda():
        for i in range(0, enc["input_ids"].size(0), batch_size):
            sl = slice(i, i + batch_size)
            out = model.generate(
                input_ids=enc["input_ids"][sl],
                attention_mask=enc.get("attention_mask", None)[sl] if enc.get("attention_mask") is not None else None,
                max_new_tokens=max_new,
                min_new_tokens=MIN_NEW_TOK,
                num_beams=beams,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                do_sample=False,
                forced_bos_token_id=lang_id,
                decoder_start_token_id=lang_id,
                use_cache=True,  # ensure fast decoding
            )
            preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return [p.strip() for p in preds]

def evaluate_pair_fast(model, tok, valid_ds, name="EN→X", limit=128):  # was 500
    src = [r["src_text"] for r in valid_ds][:limit]
    refs = [r["tgt_text"] for r in valid_ds][:limit]
    preds = batch_generate_fast(model, tok, src, max_len_src=MAX_SRC_LEN)
    refs_bleu = [[r] for r in refs]
    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = float(np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0)
    print(f"{name} -> BLEU {bleu:.2f} | chrF {chrf:.2f} | exact {exact:.2f}%  on {len(preds)} ex.")
    print("Sample preds:", list(zip(src[:2], preds[:2]))[:2])
    return {"bleu": bleu, "chrf": chrf, "exact_acc": exact}

print("🔎 Zero-shot EN→ES (before training)…")
_ = evaluate_pair_fast(mod_es, tok_es, valid_es, name="EN→ES zero-shot")

print("🔎 Zero-shot EN→PT (before training)…")
_ = evaluate_pair_fast(mod_pt, tok_pt, valid_pt, name="EN→PT zero-shot")


🔎 Zero-shot EN→ES (before training)…
EN→ES zero-shot -> BLEU 0.13 | chrF 5.33 | exact 0.00%  on 128 ex.
Sample preds: [('The patient s diet should be supplemented with multivitamins, calcium and vitamin D, and the iron and folate deficiency should be repaired.', 'y y y en en en de de de en en el el de de y en de en de y de de.'), ('The magnitude of invisibility of social vulnerability observed in the present study and its consequences for the right to comprehensive health care, reinforces the need to consider the concepts of social vulnerability and exclusion, as well as to understand the value of interdisciplinary work in the nursing practices.', 'y y y en en en de de de en en y en de en de y en y de en y y de de y de y y a de en que en en que de en, en en, y en, de en sea en en el en en.')]
🔎 Zero-shot EN→PT (before training)…
EN→PT zero-shot -> BLEU 0.00 | chrF 1.35 | exact 0.00%  on 128 ex.
Sample preds: [('Death is taboo in our culture .', 'em em em e e e'), ('Altamira, PA publish

In [ ]:
# === Cell 7: Training ===
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer
import inspect

def make_training_args(**wanted):
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in wanted.items() if k in accepted}
    dropped = sorted(set(wanted) - set(filtered))
    if dropped: print("Dropped unsupported keys:", dropped)
    return TrainingArguments(**filtered)

args_es = make_training_args(
    output_dir=f"{PROJECT_DIR}/ckpt_es",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    bf16=use_bf16, fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,
    dataloader_pin_memory=(device == "cuda"),
    evaluation_strategy="no",  # <-- FIXED KEY
    save_strategy="no",
    optim=("adamw_torch_fused" if device=="cuda" else "adamw_torch"),  # <-- safer
    group_by_length=True,
)

collator_es = DataCollatorForSeq2Seq(tokenizer=tok_es, model=mod_es, padding="longest")
trainer_es = Trainer(
    model=mod_es,
    args=args_es,
    train_dataset=train_es_tok,
    eval_dataset=None,
    data_collator=collator_es,
    tokenizer=tok_es,
)

print("✅ EN→ES trainer ready. Starting training…")
train_output_es = trainer_es.train()
print("✅ EN→ES training done.")

# Free some VRAM
import gc
del trainer_es, collator_es, train_output_es
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()

print("🔎 Evaluate EN→ES after training…")
_ = evaluate_pair(mod_es, tok_es, valid_es, name="EN→ES finetuned")


In [ ]:
# === Cell 8: PT training ===
args_pt = make_training_args(
    output_dir=f"{PROJECT_DIR}/ckpt_pt",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    bf16=use_bf16, fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,
    dataloader_pin_memory=(device == "cuda"),
    evaluation_strategy="no",
    save_strategy="no",
    optim=("adamw_torch_fused" if device=="cuda" else "adamw_torch"),
    group_by_length=True,
)

collator_pt = DataCollatorForSeq2Seq(tokenizer=tok_pt, model=mod_pt, padding="longest")
trainer_pt = Trainer(
    model=mod_pt,
    args=args_pt,
    train_dataset=train_pt_tok,
    eval_dataset=None,
    data_collator=collator_pt,
    tokenizer=tok_pt,
)

print("✅ EN→PT trainer ready. Starting training…")
train_output_pt = trainer_pt.train()
print("✅ EN→PT training done.")

# Free VRAM
import gc
del trainer_pt, collator_pt, train_output_pt
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()

print("🔎 Evaluate EN→PT after training…")
_ = evaluate_pair(mod_pt, tok_pt, valid_pt, name="EN→PT finetuned")


In [ ]:
# === Cell 9: Smoke tests ===
@torch.inference_mode()
def translate(model, tok, text, max_new_tokens=128, num_beams=4, tgt_lang=None):
    enc = tok([text], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    if tgt_lang is None:
        tgt_lang = tok.tgt_lang
    lang_id = forced_bos_id(tok, tgt_lang)
    out = model.generate(
        **enc, num_beams=num_beams, max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3, early_stopping=True,
        forced_bos_token_id=lang_id,
        decoder_start_token_id=lang_id,
    )
    return tok.batch_decode(out, skip_special_tokens=True)[0]

print("ES:", translate(mod_es, tok_es, "This is a small test of a machine translation system."))
print("PT:", translate(mod_pt, tok_pt, "How are you today?"))
